# Validacion walk-forward (5 origenes temporales, distintos regimenes hidrologicos)

Tarea de ruta critica prometida en el Anexo 1 (Fase 2, OE2): "verificando la estabilidad de los
hiperparametros elegidos en 5 origenes temporales que abarcan distintos regimenes hidrologicos".

En vez de un unico corte train/test (2019-2025 / 2026), se repite el entrenamiento y evaluacion en
5 origenes distintos, cada uno cayendo en un regimen ENSO diferente segun data/external/oni_index.csv.
Cada origen es un escenario "ventana creciente": se entrena con todo lo disponible hasta el corte, y
se evalua en el trimestre inmediatamente posterior -- asi se simula la situacion real de re-entrenar
periodicamente con el historial que se tiene en ese momento.

Se reutilizan las mismas features ya construidas en 05_features_compartidas_juan.ipynb
(dataset_features_2019_2025.csv + dataset_features_2026.csv), y la misma configuracion final
de XGBoost (06_modelo_xgboost_juan.ipynb) y Prophet (04_modelo_prophet_juan.ipynb) -- aqui no se
vuelve a tunear nada, solo se mide que tan estable es lo ya elegido.

In [1]:
# --- Celda de arranque ---
import pandas as pd
import numpy as np
from pathlib import Path

def encontrar_raiz_proyecto(marcador="requirements.txt"):
    actual = Path.cwd()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / marcador).exists():
            return carpeta
    raise FileNotFoundError(f"No encontre '{marcador}' subiendo desde {actual}")

RAIZ = encontrar_raiz_proyecto()
print("Raiz del proyecto:", RAIZ)

def calcular_metricas(y_real, y_pred):
    y_real, y_pred = np.asarray(y_real, dtype=float), np.asarray(y_pred, dtype=float)
    error = y_real - y_pred
    mae = np.abs(error).mean()
    rmse = np.sqrt((error ** 2).mean())
    mape = (np.abs(error) / y_real).mean() * 100
    return mae, rmse, mape


Raiz del proyecto: C:\Users\mgdbj\xm-spot-price-predictor


In [2]:
# --- Cargar el dataset de features ya construido (mismo insumo que 04 y 06) ---
df_train = pd.read_csv(RAIZ / "data" / "processed" / "dataset_features_2019_2025.csv", parse_dates=["fecha_hora"])
df_test = pd.read_csv(RAIZ / "data" / "processed" / "dataset_features_2026.csv", parse_dates=["fecha_hora"])

df_completo = pd.concat([df_train, df_test], ignore_index=True).sort_values("fecha_hora").reset_index(drop=True)
print("Rango disponible:", df_completo["fecha_hora"].min(), "a", df_completo["fecha_hora"].max())
print("Filas:", df_completo.shape[0])


Rango disponible: 2019-01-31 23:00:00 a 2026-08-05 23:00:00
Filas: 65833


In [3]:
# --- Los 5 origenes formales (Anexo 1) + un Origen 6 adicional que llega hasta 2026 ---
# El Origen 6 no es parte del compromiso metodologico de "5 origenes" del Anexo 1 -- se agrega
# aparte, a peticion, para ver si el patron de los otros 5 se sostiene tambien en el tramo mas
# reciente. Usa el MISMO train que el holdout oficial de 04/06/08 (2019-2025 completo), pero
# evaluado con el mismo protocolo walk-forward (mismo codigo, misma tabla) en vez de como una
# corrida aparte -- sirve ademas de control de consistencia entre ambos pipelines.
#
# Regimenes segun data/external/oni_index.csv (ONI real, no proyectado):
#   2020 Jul-Sep: -0.3 a -0.8  (La Nina, inicio)
#   2021 Jul-Sep: -0.3 a -0.6  (La Nina, continuacion)
#   2022 Oct-Dic: -0.9 a -0.7  (La Nina "triple-dip", el mas frio de la serie)
#   2023 Oct-Dic:  1.7 a  2.0  (El Nino fuerte, construyendose)
#   2024 Ene-Mar:  1.8 a  1.2  (El Nino en su pico, ya bajando)
#   2026 Ene-Ago:  -0.4 a 1.4  (El Nino 2026, de neutral a fuerte -- todo el 2026 observado)
origenes = [
    {"nombre": "Origen 1", "regimen": "La Nina (inicio)",       "corte_train": "2020-07-01", "test_inicio": "2020-07-01", "test_fin": "2020-09-30"},
    {"nombre": "Origen 2", "regimen": "La Nina (continuacion)", "corte_train": "2021-07-01", "test_inicio": "2021-07-01", "test_fin": "2021-09-30"},
    {"nombre": "Origen 3", "regimen": "La Nina (triple-dip)",   "corte_train": "2022-10-01", "test_inicio": "2022-10-01", "test_fin": "2022-12-31"},
    {"nombre": "Origen 4", "regimen": "El Nino (fuerte)",       "corte_train": "2023-10-01", "test_inicio": "2023-10-01", "test_fin": "2023-12-31"},
    {"nombre": "Origen 5", "regimen": "El Nino (pico)",         "corte_train": "2024-01-01", "test_inicio": "2024-01-01", "test_fin": "2024-03-31"},
    {"nombre": "Origen 6", "regimen": "El Nino 2026 (neutral->fuerte)", "corte_train": "2026-01-01", "test_inicio": "2026-01-01", "test_fin": "2026-08-05"},
]

for o in origenes:
    ventana = df_completo[(df_completo["fecha_hora"] >= o["test_inicio"]) & (df_completo["fecha_hora"] <= o["test_fin"])]
    o["oni_min"] = ventana["oni"].min()
    o["oni_max"] = ventana["oni"].max()
    o["n_train"] = (df_completo["fecha_hora"] < o["corte_train"]).sum()
    o["n_test"] = len(ventana)
    print(f"{o['nombre']:10s} [{o['regimen']:24s}] train<{o['corte_train']}  test {o['test_inicio']}..{o['test_fin']}  "
          f"ONI [{o['oni_min']:.1f}, {o['oni_max']:.1f}]  n_train={o['n_train']}  n_test={o['n_test']}")


Origen 1   [La Nina (inicio)        ] train<2020-07-01  test 2020-07-01..2020-09-30  ONI [-0.8, -0.2]  n_train=12385  n_test=2185
Origen 2   [La Nina (continuacion)  ] train<2021-07-01  test 2021-07-01..2021-09-30  ONI [-0.6, -0.3]  n_train=21145  n_test=2185
Origen 3   [La Nina (triple-dip)    ] train<2022-10-01  test 2022-10-01..2022-12-31  ONI [-0.9, -0.7]  n_train=32113  n_test=2185
Origen 4   [El Nino (fuerte)        ] train<2023-10-01  test 2023-10-01..2023-12-31  ONI [1.5, 2.0]  n_train=40873  n_test=2185
Origen 5   [El Nino (pico)          ] train<2024-01-01  test 2024-01-01..2024-03-31  ONI [1.2, 2.0]  n_train=43081  n_test=2161
Origen 6   [El Nino 2026 (neutral->fuerte)] train<2026-01-01  test 2026-01-01..2026-08-05  ONI [-0.6, 1.4]  n_train=60625  n_test=5185


In [4]:
# --- Mismas features que 06_modelo_xgboost_juan.ipynb (exclusion identica) ---
columnas_excluir = ["fecha_hora", "precio_bolsa", "demanda", "generacion",
                     "anio", "mes", "hora", "dia_semana", "dia_anio"]
columnas_features = [c for c in df_completo.columns if c not in columnas_excluir]

# Mismos regresores que 04_modelo_prophet_juan.ipynb (version final corregida)
columnas_regresoras_prophet = ["aportes_hidricos", "volumen_embalses", "oni", "es_pandemia", "precio_lag24h"]

print(f"{len(columnas_features)} features para XGBoost.")
print(f"{len(columnas_regresoras_prophet)} regresores para Prophet.")


33 features para XGBoost.
5 regresores para Prophet.


In [5]:
# --- Walk-forward: XGBoost (misma config final: depth=3, lr=0.01) + persistencia ---
import xgboost as xgb

resultados = []

for o in origenes:
    train = df_completo[df_completo["fecha_hora"] < o["corte_train"]].dropna(subset=columnas_features)
    test = df_completo[(df_completo["fecha_hora"] >= o["test_inicio"]) & (df_completo["fecha_hora"] <= o["test_fin"])]

    X_train, y_train = train[columnas_features], np.log(train["precio_bolsa"])
    X_test, y_test = test[columnas_features], test["precio_bolsa"]

    modelo = xgb.XGBRegressor(n_estimators=500, max_depth=3, learning_rate=0.01,
                               subsample=0.8, colsample_bytree=0.8, random_state=42)
    modelo.fit(X_train, y_train)
    y_pred = np.exp(modelo.predict(X_test))

    mae, rmse, mape = calcular_metricas(y_test, y_pred)
    resultados.append({"origen": o["nombre"], "regimen": o["regimen"], "modelo": "XGBoost",
                        "mae": mae, "rmse": rmse, "mape": mape})

    # Persistencia: precio_lag24h ya es exactamente eso (precio_bolsa desplazado 24h)
    mae_p, rmse_p, mape_p = calcular_metricas(test["precio_bolsa"], test["precio_lag24h"])
    resultados.append({"origen": o["nombre"], "regimen": o["regimen"], "modelo": "Persistencia",
                        "mae": mae_p, "rmse": rmse_p, "mape": mape_p})

    print(f"{o['nombre']} [{o['regimen']}]  XGBoost MAE {mae:6.2f} RMSE {rmse:6.2f} MAPE {mape:5.2f}%   |  "
          f"Persistencia MAE {mae_p:6.2f} RMSE {rmse_p:6.2f} MAPE {mape_p:5.2f}%")


Origen 1 [La Nina (inicio)]  XGBoost MAE  16.21 RMSE  21.46 MAPE 11.06%   |  Persistencia MAE  15.71 RMSE  22.85 MAPE 10.93%


Origen 2 [La Nina (continuacion)]  XGBoost MAE   8.44 RMSE  13.24 MAPE  8.21%   |  Persistencia MAE   6.32 RMSE  14.57 MAPE  5.60%


Origen 3 [La Nina (triple-dip)]  XGBoost MAE  43.90 RMSE  67.73 MAPE 16.40%   |  Persistencia MAE  36.07 RMSE  71.48 MAPE 15.02%


Origen 4 [El Nino (fuerte)]  XGBoost MAE 138.90 RMSE 186.28 MAPE 25.54%   |  Persistencia MAE  90.42 RMSE 142.26 MAPE 19.58%


Origen 5 [El Nino (pico)]  XGBoost MAE  46.50 RMSE  69.81 MAPE  9.79%   |  Persistencia MAE  42.49 RMSE  71.56 MAPE  9.28%


Origen 6 [El Nino 2026 (neutral->fuerte)]  XGBoost MAE  61.19 RMSE 107.45 MAPE 15.94%   |  Persistencia MAE  56.40 RMSE 112.33 MAPE 15.81%


**Prophet excluido de esta corrida.** En todas las pruebas anteriores (holdout 2026, walk-forward
original, ablaciones de interpolacion) fue consistentemente el modelo mas debil -- nunca le gano a
la persistencia, ni una sola vez, en ninguna prueba. Cada fit de Prophet toma ~20-90s dependiendo
del tamano de train; con 6 origenes el costo no se justifica dado el patron ya establecido.

In [6]:
# --- Walk-forward: ARX+GARCH (misma especificacion final que 08_modelo_arima_garch_juan.ipynb) ---
from arch import arch_model

regresoras_arx = ["aportes_hidricos", "volumen_embalses", "oni", "es_pandemia",
                   "hora_sin", "hora_cos", "dia_semana_sin", "dia_semana_cos"]
columnas_x_arx = regresoras_arx + ["precio_lag24h_log"]

for o in origenes:
    train = df_completo[df_completo["fecha_hora"] < o["corte_train"]].dropna(subset=columnas_features).copy()
    test = df_completo[(df_completo["fecha_hora"] >= o["test_inicio"]) & (df_completo["fecha_hora"] <= o["test_fin"])].copy()

    train["precio_lag24h_log"] = np.log(train["precio_lag24h"])
    test["precio_lag24h_log"] = np.log(test["precio_lag24h"])

    # Estandarizacion con estadisticos de ESTE origen unicamente (nada de 2026 ni de origenes futuros)
    y_train_log = np.log(train["precio_bolsa"])
    y_mean, y_std = y_train_log.mean(), y_train_log.std()
    y_train = (y_train_log - y_mean) / y_std * 10

    X_train_raw = train[columnas_x_arx]
    x_mean, x_std = X_train_raw.mean(), X_train_raw.std()
    X_train = (X_train_raw - x_mean) / x_std

    modelo = arch_model(y_train, x=X_train, mean="ARX", lags=0, vol="GARCH", p=1, q=1, dist="normal")
    resultado = modelo.fit(disp="off", options={"maxiter": 500})

    X_test = (test[columnas_x_arx] - x_mean) / x_std
    params_media = resultado.params[["Const"] + columnas_x_arx]
    y_pred_escalado = params_media["Const"] + (X_test * params_media[columnas_x_arx]).sum(axis=1)
    y_pred = np.exp(y_pred_escalado / 10 * y_std + y_mean)

    mae, rmse, mape = calcular_metricas(test["precio_bolsa"].values, y_pred.values)
    resultados.append({"origen": o["nombre"], "regimen": o["regimen"], "modelo": "ARX+GARCH",
                        "mae": mae, "rmse": rmse, "mape": mape})

    print(f"{o['nombre']} [{o['regimen']}]  ARX+GARCH MAE {mae:6.2f} RMSE {rmse:6.2f} MAPE {mape:5.2f}%  "
          f"(convergencia={resultado.convergence_flag})")


Origen 1 [La Nina (inicio)]  ARX+GARCH MAE  30.80 RMSE  37.76 MAPE 21.61%  (convergencia=0)


Origen 2 [La Nina (continuacion)]  ARX+GARCH MAE   7.88 RMSE  14.21 MAPE  7.44%  (convergencia=0)


Origen 3 [La Nina (triple-dip)]  ARX+GARCH MAE  44.11 RMSE  70.05 MAPE 16.86%  (convergencia=0)


C:\Users\mgdbj\AppData\Local\Temp\ipykernel_36392\3632789593.py:25: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  resultado = modelo.fit(disp="off", options={"maxiter": 500})


Origen 4 [El Nino (fuerte)]  ARX+GARCH MAE  90.04 RMSE 139.56 MAPE 19.48%  (convergencia=8)


C:\Users\mgdbj\AppData\Local\Temp\ipykernel_36392\3632789593.py:25: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  resultado = modelo.fit(disp="off", options={"maxiter": 500})


Origen 5 [El Nino (pico)]  ARX+GARCH MAE  43.73 RMSE  71.05 MAPE  9.49%  (convergencia=8)


Origen 6 [El Nino 2026 (neutral->fuerte)]  ARX+GARCH MAE  55.83 RMSE 110.08 MAPE 15.48%  (convergencia=0)


### Walk-forward: N-BEATSx y N-HiTS

Mismos 6 origenes, misma logica de reentrenar desde cero por origen (consistente con como se trato
a XGBoost y ARX+GARCH -- a diferencia de los modelos neuronales que normalmente se entrenan una vez
y se evaluan con cross_validation, aqui se prioriza la comparabilidad metodologica con el resto del
walk-forward por encima de la practica estandar de neuralforecast). Mismas exogenas y misma
configuracion que `09_modelos_deep_learning_juan.ipynb` (input_size=168h, confirmado por su propio
grid search), incluyendo la estandarizacion de exogenas -- pero calculada con SOLO los datos de
train de cada origen, para no filtrar estadisticos de un origen a otro.

In [7]:
# --- Walk-forward: N-BEATSx y N-HiTS ---
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATSx, NHITS
import time

hist_exog_dl = ["volumen_embalses", "aportes_hidricos", "demanda_lag24h"]
futr_exog_continua_dl = ["oni"]
futr_exog_ya_escalada_dl = ["hora_sin", "hora_cos", "dia_semana_sin", "dia_semana_cos"]
futr_exog_dl = futr_exog_continua_dl + futr_exog_ya_escalada_dl
INPUT_SIZE_DL = 168
MAX_STEPS_DL = 1000

for o in origenes:
    train = df_completo[df_completo["fecha_hora"] < o["corte_train"]].dropna(subset=columnas_features).copy()
    test = df_completo[(df_completo["fecha_hora"] >= o["test_inicio"]) & (df_completo["fecha_hora"] <= o["test_fin"])].copy()

    df_origen = pd.concat([train, test], ignore_index=True).sort_values("fecha_hora").reset_index(drop=True)
    mascara_train_o = df_origen["fecha_hora"] < o["corte_train"]

    df_nf_o = df_origen[["fecha_hora", "precio_bolsa"] + hist_exog_dl + futr_exog_dl].copy()
    for col in hist_exog_dl + futr_exog_continua_dl:
        mu, sigma = df_nf_o.loc[mascara_train_o, col].mean(), df_nf_o.loc[mascara_train_o, col].std()
        df_nf_o[col] = (df_nf_o[col] - mu) / sigma

    df_nf_o["unique_id"] = "precio_bolsa"
    df_nf_o = df_nf_o.rename(columns={"fecha_hora": "ds", "precio_bolsa": "y"})
    df_nf_o = df_nf_o[["unique_id", "ds", "y"] + hist_exog_dl + futr_exog_dl]

    n_test_o = int((~mascara_train_o).sum())
    n_windows_o = n_test_o // 24

    m_nbeatsx = NBEATSx(h=24, input_size=INPUT_SIZE_DL, hist_exog_list=hist_exog_dl, futr_exog_list=futr_exog_dl,
                          max_steps=MAX_STEPS_DL, val_check_steps=100, random_seed=42, enable_progress_bar=False)
    m_nhits = NHITS(h=24, input_size=INPUT_SIZE_DL, hist_exog_list=hist_exog_dl, futr_exog_list=futr_exog_dl,
                      max_steps=MAX_STEPS_DL, val_check_steps=100, random_seed=42, enable_progress_bar=False)
    nf_o = NeuralForecast(models=[m_nbeatsx, m_nhits], freq="h")

    t0 = time.time()
    cv_o = nf_o.cross_validation(df=df_nf_o, n_windows=n_windows_o, step_size=24)
    dt = time.time() - t0

    for col_modelo, nombre_modelo in [("NBEATSx", "N-BEATSx"), ("NHITS", "N-HiTS")]:
        mae, rmse, mape = calcular_metricas(cv_o["y"].values, cv_o[col_modelo].values)
        resultados.append({"origen": o["nombre"], "regimen": o["regimen"], "modelo": nombre_modelo,
                            "mae": mae, "rmse": rmse, "mape": mape})

    print(f"{o['nombre']} [{o['regimen']}]  N-BEATSx/N-HiTS listos ({dt:.0f}s, n_windows={n_windows_o})")


C:\Users\mgdbj\xm-spot-price-predictor\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-03 16:15:37,601	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-09-03 16:15:37,999	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Seed set to 42


Seed set to 42


...


  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | blocks              | ModuleList    | 4.3 M  | train
--------------------------------------------------------------
4.3 M     Trainable params
0         Non-trainable params
4.3 M     Total params
17.267    Total estimated model params size (MB)
37        Modules in train mode
0         Modules in eval mode


`Trainer.fit` stopped: `max_steps=1000` reached.


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


Origen 6 [El Nino 2026 (neutral->fuerte)]  N-BEATSx/N-HiTS listos (267s, n_windows=216)


In [8]:
# --- Tabla consolidada ---
df_resultados = pd.DataFrame(resultados)
tabla_mae = df_resultados.pivot(index=["origen", "regimen"], columns="modelo", values="mae").round(2)
tabla_mae = tabla_mae[["Persistencia", "XGBoost", "ARX+GARCH", "N-BEATSx", "N-HiTS"]]
for modelo in ["XGBoost", "ARX+GARCH", "N-BEATSx", "N-HiTS"]:
    tabla_mae[f"{modelo}_vs_persistencia"] = np.where(tabla_mae[modelo] < tabla_mae["Persistencia"], "gana", "pierde")
tabla_mae


,modelo,Persistencia,XGBoost,ARX+GARCH,N-BEATSx,N-HiTS,XGBoost_vs_persistencia,ARX+GARCH_vs_persistencia,N-BEATSx_vs_persistencia,N-HiTS_vs_persistencia
origen,regimen,,,,,,,,,
Origen 1,La Nina (inicio),15.71,16.21,30.80,17.50,19.43,pierde,pierde,pierde,pierde
Origen 2,La Nina (continuacion),6.32,8.44,7.88,5.93,6.52,pierde,pierde,gana,pierde
Origen 3,La Nina (triple-dip),36.07,43.90,44.11,38.14,38.92,pierde,pierde,pierde,pierde
Origen 4,El Nino (fuerte),90.42,138.90,90.04,75.42,78.93,pierde,gana,gana,gana
Origen 5,El Nino (pico),42.49,46.50,43.73,36.51,30.80,pierde,pierde,gana,gana
Origen 6,El Nino 2026 (neutral->fuerte),56.40,61.19,55.83,49.19,46.80,pierde,gana,gana,gana


In [9]:
# --- Estabilidad de cada modelo a lo largo de los origenes/regimenes ---
for modelo in ["Persistencia", "XGBoost", "ARX+GARCH", "N-BEATSx", "N-HiTS"]:
    maes = df_resultados[df_resultados["modelo"] == modelo]["mae"]
    cv = maes.std() / maes.mean()
    print(f"{modelo:15s} MAE medio: {maes.mean():6.2f}  desv.std: {maes.std():6.2f}  CV: {cv:.2f}")

print()
ninos = df_resultados[df_resultados["regimen"].str.contains("Nino")]
ninas = df_resultados[df_resultados["regimen"].str.contains("Nina")]
for modelo in ["Persistencia", "XGBoost", "ARX+GARCH", "N-BEATSx", "N-HiTS"]:
    mae_nino = ninos[ninos["modelo"] == modelo]["mae"].mean()
    mae_nina = ninas[ninas["modelo"] == modelo]["mae"].mean()
    print(f"{modelo:15s} MAE promedio El Nino: {mae_nino:6.2f}   MAE promedio La Nina: {mae_nina:6.2f}   razon: {mae_nino/mae_nina:.2f}x")


Persistencia    MAE medio:  41.24  desv.std:  30.16  CV: 0.73
XGBoost         MAE medio:  52.52  desv.std:  46.71  CV: 0.89
ARX+GARCH       MAE medio:  45.40  desv.std:  27.30  CV: 0.60
N-BEATSx        MAE medio:  37.11  desv.std:  24.38  CV: 0.66
N-HiTS          MAE medio:  36.90  desv.std:  25.04  CV: 0.68

Persistencia    MAE promedio El Nino:  63.10   MAE promedio La Nina:  19.37   razon: 3.26x
XGBoost         MAE promedio El Nino:  82.20   MAE promedio La Nina:  22.85   razon: 3.60x
ARX+GARCH       MAE promedio El Nino:  63.20   MAE promedio La Nina:  27.60   razon: 2.29x
N-BEATSx        MAE promedio El Nino:  53.71   MAE promedio La Nina:  20.52   razon: 2.62x
N-HiTS          MAE promedio El Nino:  52.18   MAE promedio La Nina:  21.62   razon: 2.41x


In [10]:
# --- Guardar el resultado consolidado para el informe comparativo de OE2 ---
ruta_salida = RAIZ / "data" / "processed" / "resultados" / "walkforward_5origenes.csv"
df_resultados.to_csv(ruta_salida, index=False)
print("Guardado en:", ruta_salida)


Guardado en: C:\Users\mgdbj\xm-spot-price-predictor\data\processed\resultados\walkforward_5origenes.csv
